
# Overview of Unsupervised Learning

In Supervised Learning, we played the role of a teacher: we gave the model questions ($X$) and answers ($y$).

In **Unsupervised Learning**, there is no teacher. We give the model only the data ($X$) and ask: **"What patterns can you find here?"**

## Why Use Unsupervised Learning?

1.  **Labels are expensive**: Labeling 1 million images takes humans years. Unsupervised learning can process raw data immediately.
2.  **Exploration**: Finding segments in customer data or anomalies in server logs where we don't know the ground truth.
3.  **Preprocessing**: Reducing the size/complexity of data before feeding it to a supervised model.

## Types of Unsupervised Learning

Unsupervised learning can be broadly categorized into two main types:

### Clustering (Grouping)

The goal is to separate data into groups (clusters) such that items in one group are similar to each other and different from items in other groups.

-   **K-Means**: Partitions data into K clusters by minimizing variance within each cluster. Works well for circular/spherical blobs.
-   **DBSCAN**: Groups points based on density; finds arbitrary shapes and identifies outliers.
-   **Hierarchical Clustering**: Builds a tree of clusters by merging or splitting based on distance metrics.
-   **Gaussian Mixture Models (GMM)**: Assumes data is generated from a mixture of Gaussian distributions; provides soft cluster assignments.

### Dimensionality Reduction (Simplification)

The goal is to reduce the number of features (columns) while preserving the essential **information**.

-   If you have 2 variables, "Height in cm" and "Height in inches", you don't need both. Dimensionality reduction merges them.
-   **PCA (Principal Component Analysis)**: Rotates and projects data to maximize variance along new axes.
-   **t-SNE / UMAP**: Non-linear methods for visualizing high-dimensional data in 2D/3D.
-   **Autoencoders**: Neural networks that learn compressed representations of data.

## Advantages and Disadvantages

-   **Advantages**:
    -   Can discover hidden patterns and structures in data.
    -   Useful for exploratory data analysis and feature engineering.
    -   Can handle large datasets without the need for labeled data.
-   **Disadvantages**:
    -   Results can be difficult to interpret.
    -   No guarantee of finding meaningful patterns; results are sensitive to algorithm choice and parameters.
    -   Evaluation is challenging without ground truth labels.

# Practical Demonstration: Iris Dataset

We will take the Iris dataset (4 dimensions: Sepal Length, Sepal Width, Petal Length, Petal Width) and compress it to 2 dimensions using **PCA**, then group the flowers using **K-Means**.

## Load and Inspect

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.datasets import load_iris

sns.set_theme(style="whitegrid")

# Load Data
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target  # We will NOT use this for training, only for validation later

print(f"Original Shape: {X.shape} (4 Dimensions)")

# Visualize relationships between features
df = pd.concat([X, pd.Series(y, name='species')], axis=1)
sns.pairplot(df, hue='species', palette='viridis', markers=["o", "s", "D"])
plt.suptitle('Iris Dataset Pairplot', y=1.02)
plt.tight_layout()
plt.show()

## Preprocessing (Standardization)

**Critical Step**: PCA calculates variance. If one feature is measured in "Kilometers" and another in "Millimeters", the Millimeter feature will dominate just because the numbers are bigger. We must scale everything to the same range.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Dimensionality Reduction (PCA)

We compress the 4 features into 2 "Principal Components".

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"New Shape: {X_pca.shape} (2 Dimensions)")
print(f"Variance Explained: {pca.explained_variance_ratio_}")

## Clustering (K-Means)

Now we apply K-Means to the compressed data. We tell it to look for 3 clusters (since we know there are 3 species, though in real life we use the "Elbow Method" to decide).

In [ ]:
from sklearn.cluster import KMeans

# n_init='auto' suppresses warnings in newer sklearn versions
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
kmeans.fit(X_pca)

# Get the cluster labels (0, 1, 2)
y_pred = kmeans.labels_

## Visualization and Evaluation

Let's see if the computer rediscovered the 3 species without being told what they were.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Ground Truth
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette='viridis', s=60, ax=axes[0])
axes[0].set_title("Actual Species (Ground Truth)")
axes[0].set_xlabel("PC 1")
axes[0].set_ylabel("PC 2")

# Plot 2: K-Means Clusters
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y_pred, palette='viridis', s=60, ax=axes[1])

# Plot Centroids
centers = kmeans.cluster_centers_
axes[1].scatter(centers[:, 0], centers[:, 1], c='red', s=200, marker='X', label='Centroids')

axes[1].set_title("K-Means Clusters (Unsupervised)")
axes[1].set_xlabel("PC 1")
axes[1].legend()

plt.show()

## Evaluation Metrics

Since we actually have the labels, we can measure how good the clustering was using **Adjusted Rand Index (ARI)**. If we didn't have labels, we would use **Silhouette Score** (how distinct the clusters look).

In [ ]:
from sklearn.metrics import adjusted_rand_score, silhouette_score

print(f"Adjusted Rand Index (Needs Truth): {adjusted_rand_score(y, y_pred):.2f}")
print(f"Silhouette Score (No Truth needed): {silhouette_score(X_pca, y_pred):.2f}")

## K-Means Decision Boundaries

K-Means essentially draws straight lines (Voronoi tessellation) between the centroids.

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

plt.figure(figsize=(8, 6))
DecisionBoundaryDisplay.from_estimator(
    kmeans, X_pca, response_method="predict", cmap='viridis', alpha=0.3
)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y_pred, palette='viridis', edgecolor='k')
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, marker='X')
plt.title('K-Means Decision Boundaries')
plt.show()

# Exercises: Breast Cancer Dataset

We will use the **Breast Cancer** dataset. This is a much harder problem:

-   It has **30 features** (dimensions).
-   Humans cannot visualize 30 dimensions.
-   We will use PCA to visualize it, and K-Means to see if we can separate Benign vs Malignant without labels.

## Load and Scale

## PCA Visualization

-   Use PCA to reduce the 30 features down to 2.
-   Plot the result, coloring points by the actual diagnosis.
-   **Question**: Do the classes look separable in 2D?

## Unsupervised Classification

-   Ignore the labels. Train K-Means ($K=2$) on the PCA data.
-   Check the Adjusted Rand Index against the real labels.

## Visualize Clustering Results

# Summary

1.  **Unsupervised Learning**: Finding patterns without labeled answers.
2.  **StandardScaler**: Crucial first step for distance-based algorithms (PCA, K-Means).
3.  **PCA**: Compresses data by projecting onto directions of maximum variance. Useful for visualization and speeding up training.
4.  **K-Means**: Finds groups by minimizing within-cluster variance. Simple and fast, but assumes spherical clusters.
5.  **Evaluation**: Use Silhouette Score when no labels exist; use ARI when ground truth is available.